In [0]:
# Import Libraries 
from pyspark.sql.functions import *

In [0]:
# Load Bronze Table  
crm = spark.table("crm_customers_100_k")

In [0]:
# View Data
display(crm)

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType
100000,2020-06-27,Premium,USA,57,Monthly
100001,2024-03-27,Basic,Germany,38,Annual
100002,2023-08-02,Premium,Germany,64,Annual
100003,2022-05-27,Basic,Canada,38,Annual
100004,2022-05-16,Basic,Canada,39,Monthly
100005,2024-09-13,Basic,UK,63,Annual
100006,2020-06-20,Basic,USA,45,Annual
100007,2023-10-26,Basic,Canada,48,Monthly
100008,2021-02-06,Premium,Germany,65,Monthly
100009,2020-07-07,Basic,Canada,57,Monthly


In [0]:
crm.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- SubscriptionType: string (nullable = true)



In [0]:
# Remove Duplicate Customers (One customer should appear only once) 

silver_customers = crm.dropDuplicates(["CustomerID"])

In [0]:
print("Original:", crm.count())

print("After Removing Duplicates:", silver_customers.count())

Original: 100000
After Removing Duplicates: 100000


In [0]:
# Handle Missing Values (Check missing values) 
silver_customers.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_customers.columns
]).show()

+----------+--------+-------+------+---+----------------+
|CustomerID|JoinDate|Segment|Region|Age|SubscriptionType|
+----------+--------+-------+------+---+----------------+
|         0|       0|      0|     0|  0|               0|
+----------+--------+-------+------+---+----------------+



In [0]:
# Fill missing categorical values 
silver_customers = silver_customers.fillna({
    "Segment":"Unknown",
    "Region":"Unknown",
    "SubscriptionType":"Unknown"
})

In [0]:
# Fill missing Age 
silver_customers = silver_customers.fillna({
    "Age":0
})

In [0]:
# Standardize Text(Trim spaces) 
silver_customers = silver_customers.withColumn(
    "Segment",
    trim(col("Segment"))
)

In [0]:
silver_customers = silver_customers.withColumn(
    "Region",
    trim(col("Region"))
)

In [0]:
silver_customers = silver_customers.withColumn(
    "SubscriptionType",
    trim(col("SubscriptionType"))
)

In [0]:
# Convert to uppercase 

silver_customers = silver_customers.withColumn(
    "Segment",
    upper(col("Segment"))
)

silver_customers = silver_customers.withColumn(
    "Region",
    upper(col("Region"))
)

silver_customers = silver_customers.withColumn(
    "SubscriptionType",
    upper(col("SubscriptionType"))
)

In [0]:
# Standardize Join Date (Convert string into Date) 
silver_customers = silver_customers.withColumn(
    "JoinDate",
    to_date(col("JoinDate"))
)

In [0]:
silver_customers.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- Segment: string (nullable = false)
 |-- Region: string (nullable = false)
 |-- Age: long (nullable = false)
 |-- SubscriptionType: string (nullable = false)



In [0]:
# Validate Age (Remove impossible ages) 
silver_customers = silver_customers.filter(
    (col("Age") >= 18) &
    (col("Age") <= 100)
)

In [0]:
# Create Customer Tenure (One of the most important churn features) 

silver_customers = silver_customers.withColumn(
    "TenureDays",
    datediff(current_date(), col("JoinDate"))
)

In [0]:
# Create Age Groups 
silver_customers = silver_customers.withColumn(
    "AgeGroup",
    when(col("Age")<25,"18-24")
    .when(col("Age")<35,"25-34")
    .when(col("Age")<45,"35-44")
    .when(col("Age")<60,"45-59")
    .otherwise("60+")
)

In [0]:
# Verify Output 
display(silver_customers)

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup
100000,2020-06-27,PREMIUM,USA,57,MONTHLY,2190,45-59
100001,2024-03-27,BASIC,GERMANY,38,ANNUAL,821,35-44
100002,2023-08-02,PREMIUM,GERMANY,64,ANNUAL,1059,60+
100003,2022-05-27,BASIC,CANADA,38,ANNUAL,1491,35-44
100004,2022-05-16,BASIC,CANADA,39,MONTHLY,1502,35-44
100005,2024-09-13,BASIC,UK,63,ANNUAL,651,60+
100006,2020-06-20,BASIC,USA,45,ANNUAL,2197,45-59
100007,2023-10-26,BASIC,CANADA,48,MONTHLY,974,45-59
100008,2021-02-06,PREMIUM,GERMANY,65,MONTHLY,1966,60+
100009,2020-07-07,BASIC,CANADA,57,MONTHLY,2180,45-59


In [0]:
# Save Silver Table 

silver_customers.write.mode("overwrite").saveAsTable("silver_customers")

In [0]:
# Validate 
display(spark.table("silver_customers"))

CustomerID,JoinDate,Segment,Region,Age,SubscriptionType,TenureDays,AgeGroup
100030,2022-06-19,PREMIUM,INDIA,48,MONTHLY,1468,45-59
100036,2024-09-12,BASIC,USA,43,MONTHLY,652,35-44
100042,2023-11-02,BASIC,UK,55,MONTHLY,967,45-59
100043,2021-12-10,BASIC,USA,35,MONTHLY,1659,35-44
100060,2025-01-19,BASIC,CANADA,53,MONTHLY,523,45-59
100068,2020-06-01,BASIC,UK,19,MONTHLY,2216,18-24
100082,2020-11-17,BASIC,CANADA,47,MONTHLY,2047,45-59
100084,2023-06-13,BASIC,UK,61,MONTHLY,1109,60+
100135,2021-08-30,BASIC,CANADA,60,MONTHLY,1761,60+
100140,2025-05-19,BASIC,CANADA,18,MONTHLY,403,18-24


In [0]:
# 